# WAV Chunk-Auslese und Tabelle

Dieses Notebook liest die erste WAV-Header-Sektion der Datei aus und zeigt die gefundenen RIFF-Chunks in einem pandas-DataFrame an.

In [3]:
from pathlib import Path
import pandas as pd

wav_path = Path('..') / 'data' / 'Content' / 'Native Instruments' / 'Deep Matter' / 'Samples' / 'Loops' / 'Synth' / 'Modular[122] F#m Miko 2.wav'
assert wav_path.exists(), f'WAV-Datei nicht gefunden: {wav_path}'

with wav_path.open('rb') as f:
    header = f.read(512)

assert header[:4] == b'RIFF', 'Datei ist kein RIFF/WAVE-Format'

chunks = []
junk_data = None
pos = 12
while pos + 8 <= len(header):
    chunk_id = header[pos:pos+4].decode('ascii', errors='replace')
    chunk_size = int.from_bytes(header[pos+4:pos+8], 'little')
    chunk_data = header[pos+8:pos+8+chunk_size]
    chunks.append({
        'chunk_id': chunk_id,
        'chunk_size': chunk_size,
        'chunk_pos': pos
    })
    if chunk_id == 'JUNK':
        junk_data = chunk_data
    pos += 8 + chunk_size + (chunk_size % 2)

df = pd.DataFrame(chunks)
print(df)
if junk_data is not None:
    print('\nJUNK-Chunk-Inhalt (hex):', junk_data.hex())
    print('JUNK-Chunk-Inhalt (ascii):', junk_data.decode('ascii', errors='replace'))
else:
    print('\nKein JUNK-Chunk im ausgelesenen Header gefunden.')

  chunk_id  chunk_size  chunk_pos
0     JUNK          28         12
1     fmt           16         48
2     data     2082102         72

JUNK-Chunk-Inhalt (hex): 402620000000000036c51f00000000009be20f000000000000000000
JUNK-Chunk-Inhalt (ascii): @&      6�     ��         
